In [1]:
import gcamreader
import pandas as pd
import os
from pathlib import Path
from pathlib import Path
import xml.etree.ElementTree as ET
from xml.dom import minidom

In [2]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("../")
DB_REL_PATH    = PROJECT_PATH / "output"
DB_FILE        = "database_basexdb_ERT"
QUERY_FILE     = DB_REL_PATH / "queries" / "Main_queries.xml"

REGION = ['South Korea']

In [3]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def check_query_idx():
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    for i, q in enumerate(queries):
        print(i, q.title)

def run_query(conn, q_idx, scenarios, regions=None):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    print(q.title)
    df = conn.runQuery(q, scenarios=scenarios, regions=regions)
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def get_scenario_name(conn):
    scenarios = list(conn.listScenariosInDB()['name'].unique())
    return scenarios

In [4]:
check_query_idx()

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [5]:
conn = connect_db()
get_scenario_name(conn)

Database scenarios: Baseline, Competitiveness-Erosion, Trade-Regulation, Twin-Burden, Twin-Burden-RD


['Baseline',
 'Competitiveness-Erosion',
 'Trade-Regulation',
 'Twin-Burden',
 'Twin-Burden-RD']

In [6]:
scenarios = ['Baseline', 'Competitiveness-Erosion']

In [7]:
dfCO2 = run_query(conn=conn, q_idx=265, scenarios=scenarios, regions=REGION)
dfCO2

CO2 emissions by sector (no bio) (excluding resource production)


,Units,scenario,region,sector,Year,value
0,MTC,Baseline,South Korea,H2 central production,2025,3.074355e-03
1,MTC,Baseline,South Korea,H2 central production,2030,8.137879e-02
2,MTC,Baseline,South Korea,H2 central production,2035,1.931381e-01
3,MTC,Baseline,South Korea,H2 central production,2040,3.048566e-01
4,MTC,Baseline,South Korea,H2 central production,2045,3.138861e-01
...,...,...,...,...,...,...
1685,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,2045,-1.226540e-02
1686,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,2050,-1.586002e-02
1687,MTC,Competitiveness-Erosion,South Korea,wholesale gas,2010,-8.331578e-08
1688,MTC,Competitiveness-Erosion,South Korea,wholesale gas,2015,-1.856498e-07


In [8]:
dfCO2Price = run_query(conn=conn, q_idx=297, scenarios=scenarios)
dfCO2Price

CO2 prices


,Units,scenario,Year,market,value
0,1990$/tC,Baseline,1975,EU-27CO2,0.0
1,1990$/tC,Baseline,1975,RoWCO2,0.0
2,1990$/tC,Baseline,1975,South KoreaCO2,0.0
3,1990$/tC,Baseline,1990,EU-27CO2,0.0
4,1990$/tC,Baseline,1990,RoWCO2,0.0
...,...,...,...,...,...
127,1990$/tC,Competitiveness-Erosion,2095,RoWCO2,0.0
128,1990$/tC,Competitiveness-Erosion,2095,South KoreaCO2,0.0
129,1990$/tC,Competitiveness-Erosion,2100,EU-27CO2,0.0
130,1990$/tC,Competitiveness-Erosion,2100,RoWCO2,0.0


In [9]:
CONVERT_FROM_1990_TO_2005 = 117.53 / 66.168
CONVERT_FROM_TC_TO_TCO2 = 12 / 44

In [10]:
dfCO2Price[(dfCO2Price['Year'] >= 2025) & (dfCO2Price['Year'] <= 2050)].pivot(index='Year', columns=['scenario', 'market'], values='value')

scenario Baseline                         Competitiveness-Erosion           \
market   EU-27CO2   RoWCO2 South KoreaCO2                EU-27CO2   RoWCO2   
Year                                                                         
2025      251.751    0.000        187.719                 251.751    0.000   
2030      335.600  157.332        237.785                 335.595  157.288   
2035      392.638  230.900        269.823                 392.603  230.828   
2040      422.143  281.404        308.457                 422.098  281.307   
2045      414.450  328.877        342.462                 414.276  328.751   
2050      398.927  379.541        365.956                 398.425  379.057   

scenario                 
market   South KoreaCO2  
Year                     
2025            187.719  
2030            220.834  
2035            237.444  
2040            260.696  
2045            297.039  
2050            353.486

In [11]:
dfCO2Price['value'] *= (CONVERT_FROM_1990_TO_2005 * CONVERT_FROM_TC_TO_TCO2)

In [12]:
dfCO2Price[(dfCO2Price['Year'] >= 2025) & (dfCO2Price['Year'] <= 2050)].pivot(index='Year', columns=['scenario', 'market'], values='value')

scenario    Baseline                            Competitiveness-Erosion  \
market      EU-27CO2      RoWCO2 South KoreaCO2                EU-27CO2   
Year                                                                      
2025      121.955250    0.000000      90.936352              121.955250   
2030      162.574059   76.216037     115.189728              162.571637   
2035      190.204868  111.854441     130.709834              190.187913   
2040      204.497918  136.319996     149.425228              204.476118   
2045      200.771213  159.317250     165.898205              200.686922   
2050      193.251436  183.860313     177.279358              193.008253   

scenario                             
market        RoWCO2 South KoreaCO2  
Year                                 
2025        0.000000      90.936352  
2030       76.194722     106.978188  
2035      111.819562     115.024538  
2040      136.273006     126.288459  
2045      159.256212     143.894029  
2050      183.625850     171.238540

In [13]:
dfCO2Price['Units'] = "2020$/tCO2"
dfCO2Price['pathway'] = 'ERT'
dfCO2Price[(dfCO2Price['Year'] >= 2025) & (dfCO2Price['Year'] <= 2050)].to_csv("./output/ERT_carbon_tax.csv", index=False)

In [14]:
dfSteelPrice = run_query(conn=conn, q_idx=122, scenarios=scenarios, regions=REGION)
dfSteelPrice[(dfSteelPrice['Year'] >= 2025) & (dfSteelPrice['Year'] <= 2050)].pivot(index='Year', columns='scenario', values='value')

iron and steel prices


scenario,Baseline,Competitiveness-Erosion
Year,,
2025,0.238424,0.238424
2030,0.241132,0.238890
2035,0.243425,0.239749
2040,0.247102,0.242592
2045,0.249937,0.246300
2050,0.251985,0.250547


In [15]:
us_export_share = 0.13
dfTariff = dfSteelPrice[(dfSteelPrice['Year'] >= 2025) & (dfSteelPrice['Year'] <= 2050)].pivot(index='Year', columns='scenario', values='value').copy()
dfTariff['tariff rate'] = [0, 0.5, 0.5, 0, 0, 0]
dfTariff['tariff-Baseline'] = dfTariff['Baseline'] * dfTariff['tariff rate'] * us_export_share
dfTariff['tariff-Competitiveness-Erosion'] = dfTariff['Competitiveness-Erosion'] * dfTariff['tariff rate'] * us_export_share
dfTariff

scenario,Baseline,Competitiveness-Erosion,tariff rate,tariff-Baseline,tariff-Competitiveness-Erosion
Year,,,,,
2025,0.238424,0.238424,0.0,0.000000,0.000000
2030,0.241132,0.238890,0.5,0.015674,0.015528
2035,0.243425,0.239749,0.5,0.015823,0.015584
2040,0.247102,0.242592,0.0,0.000000,0.000000
2045,0.249937,0.246300,0.0,0.000000,0.000000
2050,0.251985,0.250547,0.0,0.000000,0.000000


In [16]:
dfUsTariff = dfTariff.stack().reset_index().copy()
dfUsTariff = dfUsTariff[(dfUsTariff['Year'].isin([2030, 2035]))].copy()
dfUsTariff['pathway'] = 'ERT'
dfUsTariff.rename(columns={0: '1975$/kg'}).to_csv("./output/ERT_us_tariff.csv", index=False)

In [17]:
def build_tariff_xml(
    df,
    tariff_col="tariff-Baseline",
    region="USA",
    supplysector="traded iron and steel",
    subsector="South Korea traded iron and steel",
    technology="South Korea traded iron and steel"
):

    scenario = ET.Element("scenario")
    world = ET.SubElement(scenario, "world")
    region_el = ET.SubElement(world, "region", name=region)
    ss = ET.SubElement(region_el, "supplysector", name=supplysector)
    subsector_el = ET.SubElement(ss, "subsector", name=subsector)
    tech = ET.SubElement(subsector_el, "technology", name=technology)

    for year, row in df.iterrows():
        period = ET.SubElement(tech, "period", year=str(year))
        nie = ET.SubElement(period, "minicam-non-energy-input", name="tariff")
        cost = ET.SubElement(nie, "input-cost")
        cost.text = f"{row[tariff_col]:.4f}"

    return ET.ElementTree(scenario)

In [18]:
tree = build_tariff_xml(dfTariff, tariff_col="tariff-Baseline")
ET.indent(tree, space="    ")   # ⭐ 이 줄 하나가 핵심
tree.write(
    f"../input/policy/industry/ironsteel_tariff_ERT_Baseline.xml",
    encoding="UTF-8",
    xml_declaration=True
)

# Baseline

In [19]:
dfTax = run_query(conn=conn, q_idx=297, scenarios=['Baseline'])
dfTax = dfTax[(dfTax['Year'] <= 2050) & (dfTax['Year'] >= 2021) & (dfTax['market'].isin(['EU-27CO2', 'South KoreaCO2']))].pivot(index='Year', columns='market', values='value').copy()
dfTax['St_EU'] = [0, 0, 0.485, 1, 1, 1, 1]
dfTax['St_KOR'] = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
dfTax['P_EU_eff'] = dfTax['St_EU'] * dfTax['EU-27CO2']
dfTax['P_KOR_eff'] = dfTax['St_KOR'] * dfTax['South KoreaCO2']
dfTax['D'] = (dfTax['P_EU_eff'] - dfTax['P_KOR_eff'])
dfTax

CO2 prices


market,EU-27CO2,South KoreaCO2,St_EU,St_KOR,P_EU_eff,P_KOR_eff,D
Year,,,,,,,
2021,0.000,0.000,0.000,0.0,0.000,0.0000,0.0000
2025,251.751,187.719,0.000,0.1,0.000,18.7719,-18.7719
2030,335.600,237.785,0.485,0.2,162.766,47.5570,115.2090
2035,392.638,269.823,1.000,0.3,392.638,80.9469,311.6911
2040,422.143,308.457,1.000,0.4,422.143,123.3828,298.7602
2045,414.450,342.462,1.000,0.5,414.450,171.2310,243.2190
2050,398.927,365.956,1.000,0.6,398.927,219.5736,179.3534


In [20]:
dfCO2 = run_query(conn=conn, q_idx=268, scenarios=['Baseline'], regions=['South Korea'])
dfCO2

CO2 emissions by tech (excluding resource production)


,Units,scenario,region,sector,subsector,technology,Year,value
0,MTC,Baseline,South Korea,H2 central production,biomass,biomass to H2,2025,0.001075
1,MTC,Baseline,South Korea,H2 central production,biomass,biomass to H2,2030,0.032934
2,MTC,Baseline,South Korea,H2 central production,biomass,biomass to H2,2035,0.083659
3,MTC,Baseline,South Korea,H2 central production,biomass,biomass to H2,2040,0.147018
4,MTC,Baseline,South Korea,H2 central production,biomass,biomass to H2,2045,0.163364
...,...,...,...,...,...,...,...,...
1990,MTC,Baseline,South Korea,waste biomass for paper,biomass,biomass cogen,2030,0.153721
1991,MTC,Baseline,South Korea,waste biomass for paper,biomass,biomass cogen,2035,0.156263
1992,MTC,Baseline,South Korea,waste biomass for paper,biomass,biomass cogen,2040,0.164428
1993,MTC,Baseline,South Korea,waste biomass for paper,biomass,biomass cogen,2045,0.172183


In [21]:
arrCO2SteelDirect = dfCO2[(dfCO2['sector'].str.contains('iron and steel')) & (dfCO2['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrCO2SteelDirect

scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  17.989140
                                                BLASTFUR CCS               0.061916
                                                BLASTFUR with hydrogen     0.146227
                                                Biomass-based              0.590750
                                EAF with DRI    EAF with DRI               0.600887
                                                EAF with DRI CCS           0.006843
                                EAF with scrap  EAF with scrap             0.791416
          2035  iron and steel  BLASTFUR        BLASTFUR                  14.364250
                                                BLASTFUR CCS               0.251336
                                                BLASTFUR with hydrogen     0.378164
                                                Biomass-based              1.610240
     

In [22]:
dfElec = run_query(conn=conn, q_idx=8, scenarios=['Baseline'], regions=['South Korea'])
dfElec.groupby('Year')['value'].sum()

elec gen by subsector


Year
1990    0.358142
2005    1.313811
2010    1.696271
2015    1.879333
2021    2.076581
2025    2.199723
2030    2.383019
2035    2.598362
2040    2.836362
2045    3.051413
2050    3.155784
Name: value, dtype: float64

In [23]:
dfCO2[(dfCO2['sector'].str.contains('elec'))].groupby('Year')['value'].sum()

Year
1990     9.396560
2005    47.407164
2010    67.239118
2015    69.733274
2021    68.565066
2025    55.276003
2030    41.423475
2035    30.679653
2040    23.948543
2045    18.593319
2050    13.524348
Name: value, dtype: float64

In [24]:
arrElecI = dfCO2[(dfCO2['Year'] >= 2030) & (dfCO2['sector'].str.contains('elec'))].groupby('Year')['value'].sum() / dfElec[(dfElec['Year'] >= 2030)].groupby('Year')['value'].sum()
arrElecI

Year
2030    17.382774
2035    11.807305
2040     8.443401
2045     6.093347
2050     4.285575
Name: value, dtype: float64

In [25]:
dfH2 = run_query(conn=conn, q_idx=45, scenarios=['Baseline'], regions=['South Korea'])
dfH2.groupby('Year')['value'].sum()

hydrogen production by tech


Year
2025    0.000249
2030    0.011733
2035    0.040830
2040    0.095236
2045    0.134524
2050    0.165882
Name: value, dtype: float64

In [26]:
dfCO2[(dfCO2['sector'].str.contains('H2'))].groupby('Year')['value'].sum()

Year
2025    0.004918
2030    0.196553
2035    0.621879
2040    1.344441
2045    1.819833
2050    2.119261
Name: value, dtype: float64

In [27]:
arrH2I = dfCO2[(dfCO2['Year'] >= 2030) & (dfCO2['sector'].str.contains('H2'))].groupby('Year')['value'].sum() / dfH2[(dfH2['Year'] >= 2030)].groupby('Year')['value'].sum()
arrH2I

Year
2030    16.752412
2035    15.231101
2040    14.116967
2045    13.527964
2050    12.775682
Name: value, dtype: float64

In [28]:
dfElecI = arrElecI.reset_index()
dfElecI['input'] = 'elect_td_ind'
dfElecI

,Year,value,input
0,2030,17.382774,elect_td_ind
1,2035,11.807305,elect_td_ind
2,2040,8.443401,elect_td_ind
3,2045,6.093347,elect_td_ind
4,2050,4.285575,elect_td_ind


In [29]:
dfH2I = arrH2I.reset_index()
dfH2I['input'] = 'H2 industrial'
dfH2I

,Year,value,input
0,2030,16.752412,H2 industrial
1,2035,15.231101,H2 industrial
2,2040,14.116967,H2 industrial
3,2045,13.527964,H2 industrial
4,2050,12.775682,H2 industrial


In [30]:
df_ci = pd.concat([dfElecI, dfH2I], axis=0).rename(columns={'value': 'ci'})
df_ci

,Year,ci,input
0,2030,17.382774,elect_td_ind
1,2035,11.807305,elect_td_ind
2,2040,8.443401,elect_td_ind
3,2045,6.093347,elect_td_ind
4,2050,4.285575,elect_td_ind
0,2030,16.752412,H2 industrial
1,2035,15.231101,H2 industrial
2,2040,14.116967,H2 industrial
3,2045,13.527964,H2 industrial
4,2050,12.775682,H2 industrial


In [31]:
dfInput = run_query(conn=conn, q_idx=100, scenarios=['Baseline'], regions=['South Korea'])
dfInputIndirect = dfInput[(dfInput['sector'] == 'iron and steel') & (dfInput['input'].isin(['elect_td_ind', 'H2 industrial'])) & (dfInput['Year'] >= 2030)].copy()
df_indirect = (
    dfInputIndirect
    .merge(df_ci, on=["input", "Year"], how="left")
)
df_indirect["indirect_emissions_MtC"] = (
    df_indirect["value"] * df_indirect["ci"]
)
df_indirect
arrCO2SteelIndirect = df_indirect.groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['indirect_emissions_MtC'].sum()
arrCO2SteelIndirect

industry final energy by tech and fuel


scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  1.007778
                                                BLASTFUR CCS              0.084092
                                                BLASTFUR with hydrogen    0.032310
                                                Biomass-based             0.038198
                                EAF with DRI    EAF with DRI              0.043795
                                                EAF with DRI CCS          0.004987
                                                Hydrogen-based DRI        0.018726
                                EAF with scrap  EAF with scrap            1.772010
          2035  iron and steel  BLASTFUR        BLASTFUR                  0.546600
                                                BLASTFUR CCS              0.231867
                                                BLASTFUR with hydrogen    0.069988
                

In [32]:
arrCO2SteelDirect

scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  17.989140
                                                BLASTFUR CCS               0.061916
                                                BLASTFUR with hydrogen     0.146227
                                                Biomass-based              0.590750
                                EAF with DRI    EAF with DRI               0.600887
                                                EAF with DRI CCS           0.006843
                                EAF with scrap  EAF with scrap             0.791416
          2035  iron and steel  BLASTFUR        BLASTFUR                  14.364250
                                                BLASTFUR CCS               0.251336
                                                BLASTFUR with hydrogen     0.378164
                                                Biomass-based              1.610240
     

In [33]:
arrCO2Steel = arrCO2SteelIndirect.add(arrCO2SteelDirect, fill_value=0)
arrCO2Steel

scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  18.996918
                                                BLASTFUR CCS               0.146007
                                                BLASTFUR with hydrogen     0.178537
                                                Biomass-based              0.628948
                                EAF with DRI    EAF with DRI               0.644682
                                                EAF with DRI CCS           0.011830
                                                Hydrogen-based DRI         0.018726
                                EAF with scrap  EAF with scrap             2.563426
          2035  iron and steel  BLASTFUR        BLASTFUR                  14.910850
                                                BLASTFUR CCS               0.483204
                                                BLASTFUR with hydrogen     0.448152
     

In [34]:
dfSteelTech = run_query(conn=conn, q_idx=119, scenarios=['Baseline'], regions=['South Korea'])
arrSteelTech = dfSteelTech[(dfSteelTech['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrSteelTech

iron and steel production by tech


scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  40.330930
                                                BLASTFUR CCS               1.738790
                                                BLASTFUR with hydrogen     0.402609
                                                Biomass-based              1.528650
                                EAF with DRI    EAF with DRI               0.730706
                                                EAF with DRI CCS           0.083210
                                                Hydrogen-based DRI         0.062998
                                EAF with scrap  EAF with scrap            25.125200
          2035  iron and steel  BLASTFUR        BLASTFUR                  32.204140
                                                BLASTFUR CCS               7.058320
                                                BLASTFUR with hydrogen     1.041204
     

In [35]:
arrTax = dfTax[(dfTax.index >= 2030)]['D'] # 1990$ / tC
arrTax

Year
2030    115.2090
2035    311.6911
2040    298.7602
2045    243.2190
2050    179.3534
Name: D, dtype: float64

In [36]:
arrCO2I = (arrCO2Steel / arrSteelTech) # MtC / Mt = tC / t
arrCO2I


scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  0.471026
                                                BLASTFUR CCS              0.083971
                                                BLASTFUR with hydrogen    0.443451
                                                Biomass-based             0.411440
                                EAF with DRI    EAF with DRI              0.882273
                                                EAF with DRI CCS          0.142170
                                                Hydrogen-based DRI        0.297253
                                EAF with scrap  EAF with scrap            0.102026
          2035  iron and steel  BLASTFUR        BLASTFUR                  0.463010
                                                BLASTFUR CCS              0.068459
                                                BLASTFUR with hydrogen    0.430418
                

In [37]:
dfCo2I = (arrCO2I * 44/12).reset_index()
dfCo2I = dfCo2I[(dfCo2I['Year'].isin([2035, 2050]))].pivot(index=['subsector', 'technology'], columns='Year', values=0).reset_index()
dfCo2I['unit'] = 'tCO2 / t'
dfCo2I['sector'] = 'iron and steel'
dfCo2I.to_csv("./output/ironsteel_carbon_intensity.csv", index=False)
dfCo2I

Year,subsector,technology,2035,2050,unit,sector
0,BLASTFUR,BLASTFUR,1.697705,1.658063,tCO2 / t,iron and steel
1,BLASTFUR,BLASTFUR CCS,0.251015,0.174283,tCO2 / t,iron and steel
2,BLASTFUR,BLASTFUR with hydrogen,1.578198,1.508852,tCO2 / t,iron and steel
3,BLASTFUR,Biomass-based,1.479224,1.439578,tCO2 / t,iron and steel
4,EAF with DRI,EAF with DRI,3.164516,3.069420,tCO2 / t,iron and steel
5,EAF with DRI,EAF with DRI CCS,0.450799,0.355704,tCO2 / t,iron and steel
6,EAF with DRI,Hydrogen-based DRI,0.943285,0.724222,tCO2 / t,iron and steel
7,EAF with scrap,EAF with scrap,0.291151,0.179251,tCO2 / t,iron and steel


In [38]:
ALPHA_EU_STEEL = (2.76/63.65) # 2024 Korea Iron and Steel Association (KOSA)
CONVERT_1990_TO_1975 = 31.01 / 66.16
arrUse = ALPHA_EU_STEEL * arrCO2I * arrTax * CONVERT_1990_TO_1975 / 1000 # (tC/t) * (1990$/tC) * (1975$/1990$) / (t/kg) = (1975$/kg)
arrUse

scenario  Year  sector          subsector       technology            
Baseline  2030  iron and steel  BLASTFUR        BLASTFUR                  0.001103
                                                BLASTFUR CCS              0.000197
                                                BLASTFUR with hydrogen    0.001038
                                                Biomass-based             0.000963
                                EAF with DRI    EAF with DRI              0.002066
                                                EAF with DRI CCS          0.000333
                                                Hydrogen-based DRI        0.000696
                                EAF with scrap  EAF with scrap            0.000239
          2035  iron and steel  BLASTFUR        BLASTFUR                  0.002933
                                                BLASTFUR CCS              0.000434
                                                BLASTFUR with hydrogen    0.002727
                

In [39]:
df_use = arrUse.reset_index()
df_use.columns

Index(['scenario', 'Year', 'sector', 'subsector', 'technology', 0], dtype='object')

In [40]:
# df_use['scenario'] = 'R65'
df_use.pivot(index=['scenario', 'Year'], columns=['sector', 'technology'], values=0)

sector        iron and steel                                      \
technology          BLASTFUR BLASTFUR CCS BLASTFUR with hydrogen   
scenario Year                                                      
Baseline 2030       0.001103     0.000197               0.001038   
         2035       0.002933     0.000434               0.002727   
         2040       0.002782     0.000359               0.002562   
         2045       0.002248     0.000260               0.002059   
         2050       0.001648     0.000173               0.001500   

sector                                                                        \
technology    Biomass-based EAF with DRI EAF with DRI CCS Hydrogen-based DRI   
scenario Year                                                                  
Baseline 2030      0.000963     0.002066         0.000333           0.000696   
         2035      0.002556     0.005467         0.000779           0.001630   
         2040      0.002420     0.005170         0.000676           0.001398   
         2045      0.001954     0.004169         0.000510           0.001059   
         2050      0.001431     0.003051         0.000354           0.000720   

sector                        
technology    EAF with scrap  
scenario Year                 
Baseline 2030       0.000239  
         2035       0.000503  
         2040       0.000399  
         2045       0.000278  
         2050       0.000178

In [41]:
region_name = "South Korea"

# Filter out NaN values if needed
df_use = arrUse.reset_index()
df_use.columns = ['scenario', 'Year', 'sector', 'subsector', 'technology', 'value']
df_use = df_use[(~df_use['value'].isna()) & (df_use['subsector'] != 'biomass')].copy()

# -----------------------------------------------------------
# 1. XML ROOT: <scenario>
# -----------------------------------------------------------
root = ET.Element("scenario")

# -----------------------------------------------------------
# 2. Add <world> and <region>
# -----------------------------------------------------------
world_el = ET.SubElement(root, "world")

region_el = ET.SubElement(world_el, "region")
region_el.set("name", region_name)

In [42]:
for sector_name in df_use['sector'].unique():
    supplysector_el = ET.SubElement(region_el, "supplysector")
    supplysector_el.set("name", sector_name)

    df_sector = df_use[df_use['sector'] == sector_name]

    for subsector_name in df_sector['subsector'].unique():
        subsector_el = ET.SubElement(supplysector_el, "subsector")
        subsector_el.set("name", subsector_name)

        df_sub = df_sector[df_sector['subsector'] == subsector_name]

        for tech_name in df_sub['technology'].unique():
            tech_el = ET.SubElement(subsector_el, "stub-technology")
            tech_el.set("name", tech_name)

            df_tech = df_sub[df_sub['technology'] == tech_name]

            for _, row in df_tech.iterrows():
                year = int(row['Year'])
                val = float(row['value'])

                period_el = ET.SubElement(tech_el, "period")
                period_el.set("year", str(year))

                # NEW BLOCK:
                non_energy_el = ET.SubElement(period_el, "minicam-non-energy-input")
                non_energy_el.set("name", "IronSteel-CBAM")

                input_cost_el = ET.SubElement(non_energy_el, "input-cost")
                input_cost_el.text = f"{val:.4f}"


In [43]:
# Convert raw to string
rough = ET.tostring(root, encoding="utf-8")

# Pretty formatting
pretty = minidom.parseString(rough).toprettyxml(indent="  ")

# Save
with open("../input/policy/industry/ironsteel_cbam_ERT_Baseline.xml", "w", encoding="utf-8") as f:
    f.write(pretty)

In [44]:
arrCO2ChemDirect = dfCO2[(dfCO2['sector'].str.contains('chemical')) & (dfCO2['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrCO2ChemDirect

scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                 0.472477
                                                      biomass CCS             0.001577
                                     coal             coal                    0.082573
                                                      coal CCS                0.000064
                                     gas              gas                     0.516707
                                                      gas CCS                 0.000891
                                     refined liquids  refined liquids         1.496917
                                                      refined liquids CCS     0.002238
                chemical feedstocks  coal             coal                    0.138162
                                     refined liquids  refined liquids        11.247400
          2035  chemical energy use  biomass          bi

In [45]:
dfInput = run_query(conn=conn, q_idx=100, scenarios=['Baseline'], regions=['South Korea'])
dfInputIndirect = dfInput[(dfInput['sector'].str.contains('chemical')) & (dfInput['Year'] >= 2030)].copy()
df_indirect = (
    dfInputIndirect
    .merge(df_ci, on=["input", "Year"], how="left")
)
df_indirect["indirect_emissions_MtC"] = (
    df_indirect["value"] * df_indirect["ci"]
)
df_indirect
arrCO2ChemIndirect = df_indirect.groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['indirect_emissions_MtC'].sum()
arrCO2ChemIndirect

industry final energy by tech and fuel


scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                0.000000
                                                      biomass CCS            0.000000
                                     coal             coal                   0.000000
                                                      coal CCS               0.000000
                                     electricity      electricity            3.932086
                                     gas              gas                    0.000000
                                                      gas CCS                0.000000
                                     refined liquids  refined liquids        0.000000
                                                      refined liquids CCS    0.000000
                chemical feedstocks  coal             coal                   0.000000
                                     refined liquids  refined liqu

In [46]:
arrCO2Chem = arrCO2ChemIndirect.add(arrCO2ChemDirect, fill_value=0)
arrCO2Chem

scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                 0.472477
                                                      biomass CCS             0.001577
                                     coal             coal                    0.082573
                                                      coal CCS                0.000064
                                     electricity      electricity             3.932086
                                     gas              gas                     0.516707
                                                      gas CCS                 0.000891
                                     refined liquids  refined liquids         1.496917
                                                      refined liquids CCS     0.002238
                chemical feedstocks  coal             coal                    0.138162
                                     refined liquids  re

In [47]:
dfChemTech = run_query(conn=conn, q_idx=100, scenarios=['Baseline'], regions=['South Korea'])
arrChemTech = dfChemTech[(dfChemTech['sector'].str.contains('chemical')) & (dfChemTech['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrChemTech

industry final energy by tech and fuel


scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                0.020543
                                                      biomass CCS            0.000686
                                     coal             coal                   0.003025
                                                      coal CCS               0.000023
                                     electricity      electricity            0.226206
                                     gas              gas                    0.036388
                                                      gas CCS                0.000628
                                     refined liquids  refined liquids        0.076373
                                                      refined liquids CCS    0.001142
                chemical feedstocks  coal             coal                   0.020244
                                     refined liquids  refined liqu

In [48]:
arrTax = dfTax[(dfTax.index >= 2030)]['D'] # 1990$ / tC
arrTax

Year
2030    115.2090
2035    311.6911
2040    298.7602
2045    243.2190
2050    179.3534
Name: D, dtype: float64

In [49]:
arrCO2I = (arrCO2Chem / arrChemTech) # MTC / EJ
arrCO2I


scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                22.999981
                                                      biomass CCS             2.299996
                                     coal             coal                   27.299984
                                                      coal CCS                2.729998
                                     electricity      electricity            17.382774
                                     gas              gas                    14.200001
                                                      gas CCS                 1.420001
                                     refined liquids  refined liquids        19.600036
                                                      refined liquids CCS     1.960003
                chemical feedstocks  coal             coal                    6.825006
                                     refined liquids  re

In [50]:
dfCo2I = (arrCO2I * 44/12 / 1000).reset_index()
dfCo2I = dfCo2I[(dfCo2I['Year'].isin([2035, 2050])) & (dfCo2I['subsector'] != 'biomass')].pivot(index=['sector', 'subsector', 'technology'], columns='Year', values=0).reset_index()
dfCo2I['unit'] = 'tCO2 / GJ'
dfCo2I['sector'] = 'chemicals'
dfCo2I.to_csv("./output/chemiclas_carbon_intensity.csv", index=False)
dfCo2I

Year,sector,subsector,technology,2035,2050,unit
0,chemicals,coal,coal,0.100100,0.100100,tCO2 / GJ
1,chemicals,coal,coal CCS,0.010010,0.010010,tCO2 / GJ
2,chemicals,electricity,electricity,0.043293,0.015714,tCO2 / GJ
3,chemicals,gas,gas,0.052067,0.052067,tCO2 / GJ
4,chemicals,gas,gas CCS,0.005207,0.005207,tCO2 / GJ
5,chemicals,refined liquids,refined liquids,0.071867,0.071867,tCO2 / GJ
6,chemicals,refined liquids,refined liquids CCS,0.007187,0.007187,tCO2 / GJ
7,chemicals,coal,coal,0.025025,0.025025,tCO2 / GJ
8,chemicals,refined liquids,refined liquids,0.017967,0.017967,tCO2 / GJ


In [51]:
ALPHA_EU_CHEMICAL = (4.812/159.5)
arrUse = ALPHA_EU_CHEMICAL * arrCO2I * arrTax * CONVERT_1990_TO_1975 / 1000 # (1975$/1990$) * (1990$ / tC) * (MTC / EJ) / 1000 = 1975$ / GJ
arrUse

scenario  Year  sector               subsector        technology         
Baseline  2030  chemical energy use  biomass          biomass                0.037470
                                                      biomass CCS            0.003747
                                     coal             coal                   0.044475
                                                      coal CCS               0.004448
                                     electricity      electricity            0.028319
                                     gas              gas                    0.023134
                                                      gas CCS                0.002313
                                     refined liquids  refined liquids        0.031931
                                                      refined liquids CCS    0.003193
                chemical feedstocks  coal             coal                   0.011119
                                     refined liquids  refined liqu

In [52]:
df_use = arrUse.reset_index()
df_use.columns

Index(['scenario', 'Year', 'sector', 'subsector', 'technology', 0], dtype='object')

In [53]:
df_use['scenario'] = 'Baseline'
df_use.pivot(index=['scenario', 'Year'], columns=['sector', 'technology'], values=0)

sector        chemical energy use                                              \
technology                biomass biomass CCS      coal  coal CCS electricity   
scenario Year                                                                   
Baseline 2030            0.037470    0.003747  0.044475  0.004448    0.028319   
         2035            0.101373    0.010137  0.120326  0.012033    0.052041   
         2040            0.097168    0.009717  0.115334  0.011533    0.035671   
         2045            0.079104    0.007910  0.093893  0.009389    0.020957   
         2050            0.058332    0.005833  0.069238  0.006924    0.010869   

sector                                                                 \
technology          gas   gas CCS refined liquids refined liquids CCS   
scenario Year                                                           
Baseline 2030  0.023134  0.002313        0.031931            0.003193   
         2035  0.062587  0.006259        0.086388            0.008639   
         2040  0.059991  0.005999        0.082804            0.008280   
         2045  0.048838  0.004884        0.067410            0.006741   
         2050  0.036014  0.003601        0.049709            0.004971   

sector        chemical feedstocks                  
technology                   coal refined liquids  
scenario Year                                      
Baseline 2030            0.011119        0.007983  
         2035            0.030081        0.021597  
         2040            0.028833        0.020701  
         2045            0.023473        0.016853  
         2050            0.017309        0.012427

In [54]:
region_name = "South Korea"

# Filter out NaN values if needed
df_use = arrUse.reset_index()
df_use.columns = ['scenario', 'Year', 'sector', 'subsector', 'technology', 'value']
df_use = df_use[(~df_use['value'].isna()) & (df_use['subsector'] != 'biomass')].copy()

# -----------------------------------------------------------
# 1. XML ROOT: <scenario>
# -----------------------------------------------------------
root = ET.Element("scenario")

# -----------------------------------------------------------
# 2. Add <world> and <region>
# -----------------------------------------------------------
world_el = ET.SubElement(root, "world")

region_el = ET.SubElement(world_el, "region")
region_el.set("name", region_name)

In [55]:
for sector_name in df_use['sector'].unique():
    supplysector_el = ET.SubElement(region_el, "supplysector")
    supplysector_el.set("name", sector_name)

    df_sector = df_use[df_use['sector'] == sector_name]

    for subsector_name in df_sector['subsector'].unique():
        subsector_el = ET.SubElement(supplysector_el, "subsector")
        subsector_el.set("name", subsector_name)

        df_sub = df_sector[df_sector['subsector'] == subsector_name]

        for tech_name in df_sub['technology'].unique():
            tech_el = ET.SubElement(subsector_el, "stub-technology")
            tech_el.set("name", tech_name)

            df_tech = df_sub[df_sub['technology'] == tech_name]

            for _, row in df_tech.iterrows():
                year = int(row['Year'])
                val = float(row['value'])

                period_el = ET.SubElement(tech_el, "period")
                period_el.set("year", str(year))

                # NEW BLOCK:
                non_energy_el = ET.SubElement(period_el, "minicam-non-energy-input")
                non_energy_el.set("name", "Chemical-CBAM")

                input_cost_el = ET.SubElement(non_energy_el, "input-cost")
                input_cost_el.text = f"{val:.4f}"


In [56]:
# Convert raw to string
rough = ET.tostring(root, encoding="utf-8")

# Pretty formatting
pretty = minidom.parseString(rough).toprettyxml(indent="  ")

# Save
with open("../input/policy/industry/chemical_cbam_ERT_Baseline.xml", "w", encoding="utf-8") as f:
    f.write(pretty)

# Competitiveness Erosion

In [57]:
scenario = ['Competitiveness-Erosion']
dfTax = run_query(conn=conn, q_idx=297, scenarios=scenario)
dfTax = dfTax[(dfTax['Year'] <= 2050) & (dfTax['Year'] >= 2021) & (dfTax['market'].isin(['EU-27CO2', 'South KoreaCO2']))].pivot(index='Year', columns='market', values='value').copy()
dfTax['St_EU'] = [0, 0, 0.485, 1, 1, 1, 1]
dfTax['St_KOR'] = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
dfTax['P_EU_eff'] = dfTax['St_EU'] * dfTax['EU-27CO2']
dfTax['P_KOR_eff'] = dfTax['St_KOR'] * dfTax['South KoreaCO2']
dfTax['D'] = (dfTax['P_EU_eff'] - dfTax['P_KOR_eff'])
dfTax

CO2 prices


market,EU-27CO2,South KoreaCO2,St_EU,St_KOR,P_EU_eff,P_KOR_eff,D
Year,,,,,,,
2021,0.000,0.000,0.000,0.0,0.000000,0.0000,0.000000
2025,251.751,187.719,0.000,0.1,0.000000,18.7719,-18.771900
2030,335.595,220.834,0.485,0.2,162.763575,44.1668,118.596775
2035,392.603,237.444,1.000,0.3,392.603000,71.2332,321.369800
2040,422.098,260.696,1.000,0.4,422.098000,104.2784,317.819600
2045,414.276,297.039,1.000,0.5,414.276000,148.5195,265.756500
2050,398.425,353.486,1.000,0.6,398.425000,212.0916,186.333400


In [58]:
dfCO2 = run_query(conn=conn, q_idx=268, scenarios=scenario, regions=['South Korea'])
dfCO2

CO2 emissions by tech (excluding resource production)


,Units,scenario,region,sector,subsector,technology,Year,value
0,MTC,Competitiveness-Erosion,South Korea,H2 central production,biomass,biomass to H2,2025,0.001075
1,MTC,Competitiveness-Erosion,South Korea,H2 central production,biomass,biomass to H2,2030,0.030366
2,MTC,Competitiveness-Erosion,South Korea,H2 central production,biomass,biomass to H2,2035,0.081044
3,MTC,Competitiveness-Erosion,South Korea,H2 central production,biomass,biomass to H2,2040,0.151755
4,MTC,Competitiveness-Erosion,South Korea,H2 central production,biomass,biomass to H2,2045,0.171024
...,...,...,...,...,...,...,...,...
1982,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,biomass,biomass cogen,2030,0.154050
1983,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,biomass,biomass cogen,2035,0.156608
1984,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,biomass,biomass cogen,2040,0.164098
1985,MTC,Competitiveness-Erosion,South Korea,waste biomass for paper,biomass,biomass cogen,2045,0.170924


In [59]:
arrCO2SteelDirect = dfCO2[(dfCO2['sector'].str.contains('iron and steel')) & (dfCO2['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrCO2SteelDirect

scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  17.174070
                                                               BLASTFUR CCS               0.045053
                                                               BLASTFUR with hydrogen     0.114041
                                                               Biomass-based              0.425098
                                               EAF with DRI    EAF with DRI               0.534154
                                                               EAF with DRI CCS           0.004922
                                               EAF with scrap  EAF with scrap             0.717517
                         2035  iron and steel  BLASTFUR        BLASTFUR                  13.209090
                                                               BLASTFUR CCS               0.195884
                       

In [60]:
dfElec = run_query(conn=conn, q_idx=8, scenarios=scenario, regions=['South Korea'])
dfElec.groupby('Year')['value'].sum()

elec gen by subsector


Year
1990    0.358142
2005    1.313811
2010    1.696271
2015    1.879333
2021    2.076581
2025    2.199723
2030    2.358601
2035    2.541289
2040    2.733135
2045    2.891292
2050    2.937285
Name: value, dtype: float64

In [61]:
dfCO2[(dfCO2['sector'].str.contains('elec'))].groupby('Year')['value'].sum()

Year
1990     9.396560
2005    47.407164
2010    67.239118
2015    69.733274
2021    68.565066
2025    55.276003
2030    43.328389
2035    33.619166
2040    27.394772
2045    20.584454
2050    12.848881
Name: value, dtype: float64

In [62]:
arrElecI = dfCO2[(dfCO2['Year'] >= 2030) & (dfCO2['sector'].str.contains('elec'))].groupby('Year')['value'].sum() / dfElec[(dfElec['Year'] >= 2030)].groupby('Year')['value'].sum()
arrElecI

Year
2030    18.370376
2035    13.229176
2040    10.023203
2045     7.119466
2050     4.374407
Name: value, dtype: float64

In [63]:
dfH2 = run_query(conn=conn, q_idx=45, scenarios=scenario, regions=['South Korea'])
dfH2.groupby('Year')['value'].sum()

hydrogen production by tech


Year
2025    0.000249
2030    0.011282
2035    0.039835
2040    0.093237
2045    0.130608
2050    0.160300
Name: value, dtype: float64

In [64]:
dfCO2[(dfCO2['sector'].str.contains('H2'))].groupby('Year')['value'].sum()

Year
2025    0.004918
2030    0.191975
2035    0.623254
2040    1.371646
2045    1.852527
2050    2.141877
Name: value, dtype: float64

In [65]:
arrH2I = dfCO2[(dfCO2['Year'] >= 2030) & (dfCO2['sector'].str.contains('H2'))].groupby('Year')['value'].sum() / dfH2[(dfH2['Year'] >= 2030)].groupby('Year')['value'].sum()
arrH2I

Year
2030    17.015326
2035    15.645864
2040    14.711396
2045    14.183864
2050    13.361679
Name: value, dtype: float64

In [66]:
dfElecI = arrElecI.reset_index()
dfElecI['input'] = 'elect_td_ind'
dfElecI

,Year,value,input
0,2030,18.370376,elect_td_ind
1,2035,13.229176,elect_td_ind
2,2040,10.023203,elect_td_ind
3,2045,7.119466,elect_td_ind
4,2050,4.374407,elect_td_ind


In [67]:
dfH2I = arrH2I.reset_index()
dfH2I['input'] = 'H2 industrial'
dfH2I

,Year,value,input
0,2030,17.015326,H2 industrial
1,2035,15.645864,H2 industrial
2,2040,14.711396,H2 industrial
3,2045,14.183864,H2 industrial
4,2050,13.361679,H2 industrial


In [68]:
df_ci = pd.concat([dfElecI, dfH2I], axis=0).rename(columns={'value': 'ci'})
df_ci

,Year,ci,input
0,2030,18.370376,elect_td_ind
1,2035,13.229176,elect_td_ind
2,2040,10.023203,elect_td_ind
3,2045,7.119466,elect_td_ind
4,2050,4.374407,elect_td_ind
0,2030,17.015326,H2 industrial
1,2035,15.645864,H2 industrial
2,2040,14.711396,H2 industrial
3,2045,14.183864,H2 industrial
4,2050,13.361679,H2 industrial


In [69]:
dfInput = run_query(conn=conn, q_idx=100, scenarios=scenario, regions=['South Korea'])
dfInputIndirect = dfInput[(dfInput['sector'] == 'iron and steel') & (dfInput['input'].isin(['elect_td_ind', 'H2 industrial'])) & (dfInput['Year'] >= 2030)].copy()
df_indirect = (
    dfInputIndirect
    .merge(df_ci, on=["input", "Year"], how="left")
)
df_indirect["indirect_emissions_MtC"] = (
    df_indirect["value"] * df_indirect["ci"]
)
df_indirect
arrCO2SteelIndirect = df_indirect.groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['indirect_emissions_MtC'].sum()
arrCO2SteelIndirect

industry final energy by tech and fuel


scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  1.016781
                                                               BLASTFUR CCS              0.064665
                                                               BLASTFUR with hydrogen    0.025916
                                                               Biomass-based             0.029048
                                               EAF with DRI    EAF with DRI              0.041144
                                                               EAF with DRI CCS          0.003791
                                                               Hydrogen-based DRI        0.014773
                                               EAF with scrap  EAF with scrap            1.697823
                         2035  iron and steel  BLASTFUR        BLASTFUR                  0.563172
                                

In [70]:
arrCO2SteelDirect

scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  17.174070
                                                               BLASTFUR CCS               0.045053
                                                               BLASTFUR with hydrogen     0.114041
                                                               Biomass-based              0.425098
                                               EAF with DRI    EAF with DRI               0.534154
                                                               EAF with DRI CCS           0.004922
                                               EAF with scrap  EAF with scrap             0.717517
                         2035  iron and steel  BLASTFUR        BLASTFUR                  13.209090
                                                               BLASTFUR CCS               0.195884
                       

In [71]:
arrCO2Steel = arrCO2SteelIndirect.add(arrCO2SteelDirect, fill_value=0)
arrCO2Steel

scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  18.190851
                                                               BLASTFUR CCS               0.109718
                                                               BLASTFUR with hydrogen     0.139957
                                                               Biomass-based              0.454146
                                               EAF with DRI    EAF with DRI               0.575298
                                                               EAF with DRI CCS           0.008713
                                                               Hydrogen-based DRI         0.014773
                                               EAF with scrap  EAF with scrap             2.415340
                         2035  iron and steel  BLASTFUR        BLASTFUR                  13.772262
                       

In [72]:
dfSteelTech = run_query(conn=conn, q_idx=119, scenarios=scenario, regions=['South Korea'])
arrSteelTech = dfSteelTech[(dfSteelTech['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrSteelTech

iron and steel production by tech


scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  38.503670
                                                               BLASTFUR CCS               1.265220
                                                               BLASTFUR with hydrogen     0.313990
                                                               Biomass-based              1.100000
                                               EAF with DRI    EAF with DRI               0.649555
                                                               EAF with DRI CCS           0.059855
                                                               Hydrogen-based DRI         0.048556
                                               EAF with scrap  EAF with scrap            22.779100
                         2035  iron and steel  BLASTFUR        BLASTFUR                  29.614280
                       

In [73]:
arrTax = dfTax[(dfTax.index >= 2030)]['D'] # 1990$ / tC
arrTax

Year
2030    118.596775
2035    321.369800
2040    317.819600
2045    265.756500
2050    186.333400
Name: D, dtype: float64

In [74]:
arrCO2I = (arrCO2Steel / arrSteelTech) # MtC / Mt = tC / t
arrCO2I


scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  0.472445
                                                               BLASTFUR CCS              0.086718
                                                               BLASTFUR with hydrogen    0.445739
                                                               Biomass-based             0.412860
                                               EAF with DRI    EAF with DRI              0.885679
                                                               EAF with DRI CCS          0.145575
                                                               Hydrogen-based DRI        0.304242
                                               EAF with scrap  EAF with scrap            0.106033
                         2035  iron and steel  BLASTFUR        BLASTFUR                  0.465055
                                

In [75]:
ALPHA_EU_STEEL = (2.76/63.65) # 2024 Korea Iron and Steel Association (KOSA)
CONVERT_1990_TO_1975 = 31.01 / 66.16
arrUse = ALPHA_EU_STEEL * arrCO2I * arrTax * CONVERT_1990_TO_1975 / 1000 # (tC/t) * (1990$/tC) * (1975$/1990$) / (t/kg) = (1975$/kg)
arrUse

scenario                 Year  sector          subsector       technology            
Competitiveness-Erosion  2030  iron and steel  BLASTFUR        BLASTFUR                  0.001139
                                                               BLASTFUR CCS              0.000209
                                                               BLASTFUR with hydrogen    0.001074
                                                               Biomass-based             0.000995
                                               EAF with DRI    EAF with DRI              0.002135
                                                               EAF with DRI CCS          0.000351
                                                               Hydrogen-based DRI        0.000733
                                               EAF with scrap  EAF with scrap            0.000256
                         2035  iron and steel  BLASTFUR        BLASTFUR                  0.003038
                                

In [76]:
df_use = arrUse.reset_index()
df_use.columns

Index(['scenario', 'Year', 'sector', 'subsector', 'technology', 0], dtype='object')

In [77]:
# df_use['scenario'] = 'R65'
df_use.pivot(index=['scenario', 'Year'], columns=['sector', 'technology'], values=0)

sector                       iron and steel               \
technology                         BLASTFUR BLASTFUR CCS   
scenario                Year                               
Competitiveness-Erosion 2030       0.001139     0.000209   
                        2035       0.003038     0.000473   
                        2040       0.002974     0.000410   
                        2045       0.002464     0.000299   
                        2050       0.001713     0.000181   

sector                                                             \
technology                   BLASTFUR with hydrogen Biomass-based   
scenario                Year                                        
Competitiveness-Erosion 2030               0.001074      0.000995   
                        2035               0.002834      0.002648   
                        2040               0.002753      0.002589   
                        2045               0.002270      0.002143   
                        2050               0.001566      0.001487   

sector                                                                         \
technology                   EAF with DRI EAF with DRI CCS Hydrogen-based DRI   
scenario                Year                                                    
Competitiveness-Erosion 2030     0.002135         0.000351           0.000733   
                        2035     0.005669         0.000835           0.001749   
                        2040     0.005535         0.000754           0.001576   
                        2045     0.004574         0.000577           0.001226   
                        2050     0.003171         0.000369           0.000781   

sector                                       
technology                   EAF with scrap  
scenario                Year                 
Competitiveness-Erosion 2030       0.000256  
                        2035       0.000556  
                        2040       0.000466  
                        2045       0.000326  
                        2050       0.000187

In [78]:
region_name = "South Korea"

# Filter out NaN values if needed
df_use = arrUse.reset_index()
df_use.columns = ['scenario', 'Year', 'sector', 'subsector', 'technology', 'value']
df_use = df_use[(~df_use['value'].isna()) & (df_use['subsector'] != 'biomass')].copy()

# -----------------------------------------------------------
# 1. XML ROOT: <scenario>
# -----------------------------------------------------------
root = ET.Element("scenario")

# -----------------------------------------------------------
# 2. Add <world> and <region>
# -----------------------------------------------------------
world_el = ET.SubElement(root, "world")

region_el = ET.SubElement(world_el, "region")
region_el.set("name", region_name)

In [79]:
for sector_name in df_use['sector'].unique():
    supplysector_el = ET.SubElement(region_el, "supplysector")
    supplysector_el.set("name", sector_name)

    df_sector = df_use[df_use['sector'] == sector_name]

    for subsector_name in df_sector['subsector'].unique():
        subsector_el = ET.SubElement(supplysector_el, "subsector")
        subsector_el.set("name", subsector_name)

        df_sub = df_sector[df_sector['subsector'] == subsector_name]

        for tech_name in df_sub['technology'].unique():
            tech_el = ET.SubElement(subsector_el, "stub-technology")
            tech_el.set("name", tech_name)

            df_tech = df_sub[df_sub['technology'] == tech_name]

            for _, row in df_tech.iterrows():
                year = int(row['Year'])
                val = float(row['value'])

                period_el = ET.SubElement(tech_el, "period")
                period_el.set("year", str(year))

                # NEW BLOCK:
                non_energy_el = ET.SubElement(period_el, "minicam-non-energy-input")
                non_energy_el.set("name", "IronSteel-CBAM")

                input_cost_el = ET.SubElement(non_energy_el, "input-cost")
                input_cost_el.text = f"{val:.4f}"


In [80]:
# Convert raw to string
rough = ET.tostring(root, encoding="utf-8")

# Pretty formatting
pretty = minidom.parseString(rough).toprettyxml(indent="  ")

# Save
with open("../input/policy/industry/ironsteel_cbam_ERT_Competitiveness_Erosion.xml", "w", encoding="utf-8") as f:
    f.write(pretty)

In [81]:
arrCO2ChemDirect = dfCO2[(dfCO2['sector'].str.contains('chemical')) & (dfCO2['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrCO2ChemDirect

scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                 0.453590
                                                                     biomass CCS             0.000901
                                                    coal             coal                    0.081968
                                                                     coal CCS                0.000040
                                                    gas              gas                     0.506547
                                                                     gas CCS                 0.000584
                                                    refined liquids  refined liquids         1.474630
                                                                     refined liquids CCS     0.001499
                               chemical feedstocks  coal             coal                    0.

In [82]:
dfInput = run_query(conn=conn, q_idx=100, scenarios=scenario, regions=['South Korea'])
dfInputIndirect = dfInput[(dfInput['sector'].str.contains('chemical')) & (dfInput['Year'] >= 2030)].copy()
df_indirect = (
    dfInputIndirect
    .merge(df_ci, on=["input", "Year"], how="left")
)
df_indirect["indirect_emissions_MtC"] = (
    df_indirect["value"] * df_indirect["ci"]
)
df_indirect
arrCO2ChemIndirect = df_indirect.groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['indirect_emissions_MtC'].sum()
arrCO2ChemIndirect

industry final energy by tech and fuel


scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                0.000000
                                                                     biomass CCS            0.000000
                                                    coal             coal                   0.000000
                                                                     coal CCS               0.000000
                                                    electricity      electricity            3.972307
                                                    gas              gas                    0.000000
                                                                     gas CCS                0.000000
                                                    refined liquids  refined liquids        0.000000
                                                                     refined liquids CCS    0.000000
  

In [83]:
arrCO2Chem = arrCO2ChemIndirect.add(arrCO2ChemDirect, fill_value=0)
arrCO2Chem

scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                 0.453590
                                                                     biomass CCS             0.000901
                                                    coal             coal                    0.081968
                                                                     coal CCS                0.000040
                                                    electricity      electricity             3.972307
                                                    gas              gas                     0.506547
                                                                     gas CCS                 0.000584
                                                    refined liquids  refined liquids         1.474630
                                                                     refined liquids CCS     0.

In [84]:
dfChemTech = run_query(conn=conn, q_idx=100, scenarios=scenario, regions=['South Korea'])
arrChemTech = dfChemTech[(dfChemTech['sector'].str.contains('chemical')) & (dfChemTech['Year'] >= 2030)].groupby(['scenario', 'Year', 'sector', 'subsector', 'technology'])['value'].sum()
arrChemTech

industry final energy by tech and fuel


scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                0.019721
                                                                     biomass CCS            0.000392
                                                    coal             coal                   0.003002
                                                                     coal CCS               0.000015
                                                    electricity      electricity            0.216234
                                                    gas              gas                    0.035672
                                                                     gas CCS                0.000411
                                                    refined liquids  refined liquids        0.075237
                                                                     refined liquids CCS    0.000765
  

In [85]:
arrTax = dfTax[(dfTax.index >= 2030)]['D'] # 1990$ / tC
arrTax

Year
2030    118.596775
2035    321.369800
2040    317.819600
2045    265.756500
2050    186.333400
Name: D, dtype: float64

In [86]:
arrCO2I = (arrCO2Chem / arrChemTech) # MTC / EJ
arrCO2I


scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                22.999973
                                                                     biomass CCS             2.300000
                                                    coal             coal                   27.300012
                                                                     coal CCS                2.730000
                                                    electricity      electricity            18.370376
                                                    gas              gas                    14.200015
                                                                     gas CCS                 1.419999
                                                    refined liquids  refined liquids        19.599923
                                                                     refined liquids CCS     1.

In [87]:
ALPHA_EU_CHEMICAL = (4.812/159.5)
arrUse = ALPHA_EU_CHEMICAL * arrCO2I * arrTax * CONVERT_1990_TO_1975 / 1000 # (1975$/1990$) * (1990$ / tC) * (MTC / EJ) / 1000 = 1975$ / GJ
arrUse

scenario                 Year  sector               subsector        technology         
Competitiveness-Erosion  2030  chemical energy use  biomass          biomass                0.038572
                                                                     biomass CCS            0.003857
                                                    coal             coal                   0.045783
                                                                     coal CCS               0.004578
                                                    electricity      electricity            0.030808
                                                    gas              gas                    0.023814
                                                                     gas CCS                0.002381
                                                    refined liquids  refined liquids        0.032870
                                                                     refined liquids CCS    0.003287
  

In [88]:
df_use = arrUse.reset_index()
df_use.columns

Index(['scenario', 'Year', 'sector', 'subsector', 'technology', 0], dtype='object')

In [89]:
df_use['scenario'] = scenario[0]
df_use.pivot(index=['scenario', 'Year'], columns=['sector', 'technology'], values=0)

sector                       chemical energy use                        \
technology                               biomass biomass CCS      coal   
scenario                Year                                             
Competitiveness-Erosion 2030            0.038572    0.003857  0.045783   
                        2035            0.104521    0.010452  0.124062   
                        2040            0.103367    0.010337  0.122691   
                        2045            0.086434    0.008643  0.102593   
                        2050            0.060602    0.006060  0.071932   

sector                                                                  \
technology                    coal CCS electricity       gas   gas CCS   
scenario                Year                                             
Competitiveness-Erosion 2030  0.004578    0.030808  0.023814  0.002381   
                        2035  0.012406    0.060119  0.064530  0.006453   
                        2040  0.012269    0.045046  0.063818  0.006382   
                        2045  0.010259    0.026755  0.053363  0.005336   
                        2050  0.007193    0.011526  0.037415  0.003742   

sector                                                            \
technology                   refined liquids refined liquids CCS   
scenario                Year                                       
Competitiveness-Erosion 2030        0.032870            0.003287   
                        2035        0.089070            0.008907   
                        2040        0.088086            0.008809   
                        2045        0.073657            0.007366   
                        2050        0.051644            0.005164   

sector                       chemical feedstocks                  
technology                                  coal refined liquids  
scenario                Year                                      
Competitiveness-Erosion 2030            0.011446        0.008218  
                        2035            0.031015        0.022268  
                        2040            0.030673        0.022022  
                        2045            0.025648        0.018414  
                        2050            0.017983        0.012911

In [90]:
region_name = "South Korea"

# Filter out NaN values if needed
df_use = arrUse.reset_index()
df_use.columns = ['scenario', 'Year', 'sector', 'subsector', 'technology', 'value']
df_use = df_use[(~df_use['value'].isna()) & (df_use['subsector'] != 'biomass')].copy()

# -----------------------------------------------------------
# 1. XML ROOT: <scenario>
# -----------------------------------------------------------
root = ET.Element("scenario")

# -----------------------------------------------------------
# 2. Add <world> and <region>
# -----------------------------------------------------------
world_el = ET.SubElement(root, "world")

region_el = ET.SubElement(world_el, "region")
region_el.set("name", region_name)

In [91]:
for sector_name in df_use['sector'].unique():
    supplysector_el = ET.SubElement(region_el, "supplysector")
    supplysector_el.set("name", sector_name)

    df_sector = df_use[df_use['sector'] == sector_name]

    for subsector_name in df_sector['subsector'].unique():
        subsector_el = ET.SubElement(supplysector_el, "subsector")
        subsector_el.set("name", subsector_name)

        df_sub = df_sector[df_sector['subsector'] == subsector_name]

        for tech_name in df_sub['technology'].unique():
            tech_el = ET.SubElement(subsector_el, "stub-technology")
            tech_el.set("name", tech_name)

            df_tech = df_sub[df_sub['technology'] == tech_name]

            for _, row in df_tech.iterrows():
                year = int(row['Year'])
                val = float(row['value'])

                period_el = ET.SubElement(tech_el, "period")
                period_el.set("year", str(year))

                # NEW BLOCK:
                non_energy_el = ET.SubElement(period_el, "minicam-non-energy-input")
                non_energy_el.set("name", "Chemical-CBAM")

                input_cost_el = ET.SubElement(non_energy_el, "input-cost")
                input_cost_el.text = f"{val:.4f}"


In [92]:
# Convert raw to string
rough = ET.tostring(root, encoding="utf-8")

# Pretty formatting
pretty = minidom.parseString(rough).toprettyxml(indent="  ")

# Save
with open("../input/policy/industry/chemical_cbam_ERT_Competitiveness_Erosion.xml", "w", encoding="utf-8") as f:
    f.write(pretty)